# M1: Validación de Detección de Placas + OCR
Sube ~30 imágenes en `/content/drive/MyDrive/peaje/validacion_placas/` nombradas `1234ABC.jpg` o `1234ABC_1.jpg`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/peaje/validacion_placas /content/drive/MyDrive/peaje/evidencias
!git clone https://github.com/Adiel05/Control_Peaje.git /content/Control_Peaje 2>/dev/null || git -C /content/Control_Peaje pull
%cd /content/Control_Peaje
!pip -q install -r inferencia/requirements.txt


In [ ]:
import csv, glob, json, os, cv2
from inferencia.m1_placas import LectorPlacas, limpiar_placa

lector = LectorPlacas()
imagenes = sorted(glob.glob('/content/drive/MyDrive/peaje/validacion_placas/*.*'))
filas, detectadas, exactas, chars_ok, chars_tot = [], 0, 0, 0, 0

for ruta in imagenes:
    nombre = os.path.splitext(os.path.basename(ruta))[0].split('_')[0]
    gt = limpiar_placa(nombre)
    img = cv2.imread(ruta)
    if img is None:
        continue
    res = lector.leer(img)
    pred = res['placa']
    if res['caja'] is not None:
        detectadas += 1
    if pred == gt:
        exactas += 1
    n_max = max(len(gt), len(pred))
    chars_tot += n_max
    chars_ok += sum(1 for a, b in zip(gt, pred) if a == b)
    filas.append({'archivo': os.path.basename(ruta), 'gt': gt, 'pred': pred, 'confianza': res['confianza'], 'exacta': pred == gt})

total = len(filas)
resumen = {
    'total_imagenes': total,
    'tasa_deteccion': detectadas / total if total else 0.0,
    'exactitud_placa': exactas / total if total else 0.0,
    'exactitud_caracter': chars_ok / chars_tot if chars_tot else 0.0,
}
print(json.dumps(resumen, indent=2))
with open('/content/drive/MyDrive/peaje/evidencias/metricas_m1.json', 'w') as f:
    json.dump(resumen, f, indent=2)
if filas:
    with open('/content/drive/MyDrive/peaje/evidencias/metricas_m1_ocr.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=filas[0].keys())
        w.writeheader(); w.writerows(filas)
